# AWP Data — Universal Agent Workflow

This notebook demonstrates `awp.data.AgentWorkflow` — a programmatic API
that wraps AWP's A4 delegation loop engine. Pass arbitrary inputs + a task,
and get results back as JSON.

**No YAML workflow files required.**

## Setup

```bash
pip install -e "reference/python/[data]"
```

The next cell lets you choose your LLM provider. Supported options:
- **Ollama** — local, no API key needed (install from ollama.com)
- **OpenRouter** — cloud, requires `OPENROUTER_API_KEY`
- **Custom API** — any OpenAI-compatible endpoint, requires `LLM_API_KEY` + `LLM_BASE_URL`

In [1]:
# ============================================================
# Provider Selection — choose ONE of: "ollama", "openrouter", "custom"
# ============================================================
PROVIDER = "openrouter"  # <-- change this

# --- Ollama (local) -------------------------------------------
OLLAMA_MODEL = "qwen3:1.7b"            # any model you've pulled
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# --- OpenRouter (cloud) ---------------------------------------
OPENROUTER_API_KEY = "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"            # paste your key or set env var OPENROUTER_API_KEY
OPENROUTER_MODEL = "nvidia/nemotron-3-super-120b-a12b:free"

# --- Custom OpenAI-compatible API -----------------------------
CUSTOM_API_KEY = ""                     # paste your key or set env var LLM_API_KEY
CUSTOM_BASE_URL = ""                    # e.g. "https://api.openai.com/v1"
CUSTOM_MODEL = ""                       # e.g. "gpt-4o"

# ==============================================================
# DO NOT EDIT BELOW — wires up the selected provider
# ==============================================================
import os

if PROVIDER == "ollama":
    os.environ["LLM_API_KEY"] = "ollama"
    os.environ["LLM_BASE_URL"] = OLLAMA_BASE_URL
    MODEL = f"ollama/{OLLAMA_MODEL}"
    print(f"✔ Using Ollama  model={OLLAMA_MODEL}  url={OLLAMA_BASE_URL}")

elif PROVIDER == "openrouter":
    key = OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY", "")
    if not key:
        raise ValueError("Set OPENROUTER_API_KEY above or as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
    MODEL = OPENROUTER_MODEL
    print(f"✔ Using OpenRouter  model={OPENROUTER_MODEL}")

elif PROVIDER == "custom":
    key = CUSTOM_API_KEY or os.getenv("LLM_API_KEY", "")
    url = CUSTOM_BASE_URL or os.getenv("LLM_BASE_URL", "")
    if not key:
        raise ValueError("Set CUSTOM_API_KEY above or LLM_API_KEY as an environment variable")
    if not url:
        raise ValueError("Set CUSTOM_BASE_URL above or LLM_BASE_URL as an environment variable")
    os.environ["LLM_API_KEY"] = key
    os.environ["LLM_BASE_URL"] = url
    MODEL = CUSTOM_MODEL
    print(f"✔ Using custom API  model={CUSTOM_MODEL}  url={url}")

else:
    raise ValueError(f"Unknown PROVIDER '{PROVIDER}'. Use 'ollama', 'openrouter', or 'custom'.")

# Now import AWP (after env vars are set)
import pandas as pd
from awp.data import AgentWorkflow

✔ Using OpenRouter  model=nvidia/nemotron-3-super-120b-a12b:free


## Example 1: DataFrame Analysis

Pass a pandas DataFrame and ask the agent to analyze it.

In [2]:
# Create sample data
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=100, freq="D"),
    "revenue": [100 + i * 2.5 + (i % 7) * 10 for i in range(100)],
    "users": [500 + i * 5 + (i % 3) * 20 for i in range(100)],
    "region": ["EU", "US", "APAC", "EU", "US"] * 20,
})
df.head()

,date,revenue,users,region
0,2024-01-01,100.0,500,EU
1,2024-01-02,112.5,525,US
2,2024-01-03,125.0,550,APAC
3,2024-01-04,137.5,515,EU
4,2024-01-05,150.0,540,US


In [3]:
result = AgentWorkflow(
    inputs={
        "sales_data": df,
    },
    task="Analyze the sales data: find trends per region, calculate growth rates, and summarize key insights.",
    model=MODEL,
    max_loops=5,
    max_total_tokens=200_000,
    max_wall_time=600,
    output_dir="./output_example1",
    verbose=True,
).run()

print(f"Status: {result['status']}")
print(f"Metadata: {result['metadata']}")
print(f"Artifacts: {result['artifacts']}")

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example1
INFO:awp.data.inputs:Input 'sales_data': DataFrame -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example1/workspace/inputs/sales_data.csv
INFO:awp.data.workflow:Starting delegation loop: task=Analyze the sales data: find trends per region, calculate growth rates, and summ
INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_01-50-39_687235d3] depth=0 starting: Analyze the sales data: find trends per region, calculate growth rates, and summ
INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===
DEBUG:awp.runtime.llm:LLM request: model=nvidia/nemotron-3-super-120b-a12b:free, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:awp.runtime.llm:LLM request: model=nvidia/nemotron-3-super-120b-a12b:free, messages=2, tools=0
INFO:httpx:HTTP Request:


  AWP DELEGATION LOOP DEBUG REPORT
  Model:         nvidia/nemotron-3-super-120b-a12b:free
  Worker model:  nvidia/nemotron-3-super-120b-a12b:free
  Budget:        loops=5, workers=30, tokens=200,000, wall_time=600s, depth=5

  ────────────────────────────────────────────────────────
  Iteration 001
  ────────────────────────────────────────────────────────
    ────────────────────────────────────────────────
    MANAGER DECISION
    ────────────────────────────────────────────────
    Decision:    delegate
    Reasoning:
      | The task requires loading sales data, computing regional trends, calculating growth rates, and summarizing insights. A single worker with file reading capability can perform these analyses using its internal reasoning, so we delegate to one specialized worker.
    Full Manager Decision JSON:
      {
        "decision": "delegate",
        "reasoning": "The task requires loading sales data, computing regional trends, calculating growth rates, and summarizing i

In [4]:
# Inspect the result
import json
print(json.dumps(result["result"], indent=2, default=str))

{
  "error": "Wall time exceeded (642.3s > 600s)",
  "partial_result": {},
  "confidence": 0.0
}


## Example 2: Mixed Inputs

Pass multiple input types: DataFrame, config dict, and a text description.

In [5]:
result2 = AgentWorkflow(
    inputs={
        "data": df,
        "config": {
            "target_metric": "revenue",
            "threshold": 0.8,
            "regions_of_interest": ["EU", "US"],
        },
        "context": "We are preparing a quarterly business review for stakeholders.",
    },
    task="Create a summary report focusing on the target metric for specified regions. Include growth trends.",
    model=MODEL,
    max_loops=5,
    max_wall_time=600,
    output_dir="./output_example2",
).run()

print(f"Status: {result2['status']}")
print(json.dumps(result2["result"], indent=2, default=str))

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example2
INFO:awp.data.inputs:Input 'data': DataFrame -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example2/workspace/inputs/data.csv
INFO:awp.data.inputs:Input 'config': Dict -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example2/workspace/inputs/config.json
INFO:awp.data.workflow:Starting delegation loop: task=Create a summary report focusing on the target metric for specified regions. Inc
INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_02-01-38_7f46d52d] depth=0 starting: Create a summary report focusing on the target metric for specified regions. Inc
INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===
DEBUG:awp.runtime.llm:LLM request: model=nvidia/nemotron-3-super-120b-a12b:free, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "

Status: error
{
  "error": "The task does not specify the target metric or the regions to analyze, which are required to generate a meaningful summary report with growth trends.",
  "partial_result": {},
  "confidence": 0.0
}


## Example 3: File Path Input

Pass a file path instead of in-memory data.

In [6]:
# Save DataFrame to CSV first
df.to_csv("/tmp/sales_data.csv", index=False)

result3 = AgentWorkflow(
    inputs={
        "csv_file": "/tmp/sales_data.csv",
    },
    task="Load the CSV file and calculate the average revenue per region.",
    model=MODEL,
    max_loops=3,
    max_wall_time=60,
    output_dir="./output_example3",
).run()

print(f"Status: {result3['status']}")
print(json.dumps(result3["result"], indent=2, default=str))

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example3
INFO:awp.data.inputs:Input 'csv_file': File -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_example3/workspace/inputs/sales_data.csv
INFO:awp.data.workflow:Starting delegation loop: task=Load the CSV file and calculate the average revenue per region.
INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-28_02-02-50_36ae6606] depth=0 starting: Load the CSV file and calculate the average revenue per region.
INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===
DEBUG:awp.runtime.llm:LLM request: model=nvidia/nemotron-3-super-120b-a12b:free, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
DEBUG:awp.runtime.llm:LLM request: model=nvidia/nemotron-3-super-120b-a12b:free, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/c

Status: partial
{
  "partial": true,
  "termination_reason": "max_wall_time exceeded",
  "iterations_completed": 1,
  "confidence": 0.5,
  "history_summary": [
    {
      "iteration": 1,
      "confidence": 0.5
    }
  ]
}


## Budget & Parameter Reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `model` | *(required)* | LLM model identifier |
| `worker_model` | = `model` | Model for worker agents |
| `max_loops` | 10 | Max delegation loop iterations |
| `max_total_tokens` | 500,000 | Max total LLM tokens |
| `max_wall_time` | 300 | Max wall time (seconds) |
| `max_tool_calls` | 100 | Max tool invocations |
| `max_total_workers` | 30 | Max worker agents spawned |
| `max_depth` | 5 | Max recursive delegation depth |
| `sandbox` | "subprocess" | Sandbox type: subprocess/docker/venv/none |
| `packages` | [] | Extra pip packages for sandbox |
| `output_dir` | *(temp)* | Output directory for artifacts |
| `verbose` | False | Enable debug logging |